# Prefix Reuse and KMP


```{contents}
:local:
:depth: 2
```


```{index} Reuse a Prefix Only When It Is Also the Needed Suffix
```



```{index} Build the Fallback Table Before Scanning Text
```

## Build the Fallback Table Before Scanning Text


KMP avoids restarting a search from scratch after every mismatch. It retains a fact already proved: a suffix of the consumed text matches a prefix of the pattern. This connects direct matching's result contract to a different reuse rule. The goal remains all overlapping ordinal matches; changing the algorithm must not change that answer.

For each pattern position i, prefix[i] records the longest **proper** prefix of pattern[0..i] that is also its suffix. Proper excludes the entire segment. The number is a length, not an index of the next text character. For ABABACA, complete the table from left to right:

| i | Current segment | Border retained | prefix[i] |
| --- | --- | --- | ---: |
| 0 | A | Empty | 0 |
| 1 | AB | Empty | 0 |
| 2 | ABA | A | 1 |
| 3 | ABAB | AB | 2 |
| 4 | ABABA | ABA | 3 |
| 5 | ABABAC | Empty | 0 |
| 6 | ABABACA | A | 1 |

At i=5 the candidate length is three, so the next expected character is pattern[3], B. C mismatches. Follow prefix[2] to candidate length one; pattern[1] is B, so C still mismatches. Follow prefix[0] to zero; pattern[0] is A, also a mismatch. Store zero. This is why a fallback can happen several times at one position without moving that position backwards.

The core schematic update is `q = prefix[q - 1]` when q>0 and the next character fails. q is the number already matched; q-1 is the last position of that matched prefix. Using prefix[q] instead confuses the current known prefix with a longer segment that was never established.


In [1]:
using System;
static int[] Prefix(string pattern)
{
    int[] prefix = new int[pattern.Length];
    int q = 0;
    for (int i = 1; i < pattern.Length; i++)
    {
        while (q > 0 && pattern[i] != pattern[q]) q = prefix[q - 1];
        if (pattern[i] == pattern[q]) q++;
        prefix[i] = q;
    }
    return prefix;
}
Console.WriteLine(string.Join(",", Prefix("ABABACA")));
Console.WriteLine(string.Join(",", Prefix("AAAA")));


0,0,1,2,3,0,1
0,1,2,3


For AAAA, each new character extends the preceding border by one, but the final border has length three, not four. An empty pattern produces an empty table because the loop does not execute. The scanner handles that pattern before attempting to index it.


**Common mistake.** Letting the border include the whole segment. `prefix[i]` must be a *proper* prefix length, strictly less than `i + 1`; otherwise `AAAA` would produce `1, 2, 3, 4` and the scan would never shift.


```{index} Keep a Proven Suffix After Reporting a Match
```

## Keep a Proven Suffix After Reporting a Match


Use pattern ABAB and text ABABAB. Its prefix table is 0,0,1,2. The scan keeps i as the consumed text position and q as the matched-prefix length.

| Consumed text position | Character | q before comparison | q after a match | Report/fallback |
| --- | --- | ---: | ---: | --- |
| 0 | A | 0 | 1 | None |
| 1 | B | 1 | 2 | None |
| 2 | A | 2 | 3 | None |
| 3 | B | 3 | 4 | Start 0; retain q=2 |
| 4 | A | 2 | 3 | None |
| 5 | B | 3 | 4 | Start 2; retain q=2 |

After the first match, AB at the end of the matched text is already a valid beginning of another occurrence. Keeping q=2 preserves the overlap at start two. Resetting q to zero would lose that occurrence. The focused scanner below repeats its own setup so it can run independently of the prefix-table cell.


In [1]:
using System;
using System.Collections.Generic;
static int[] AllMatches(string text, string pattern)
{
    if (pattern.Length == 0)
    {
        int[] boundaries = new int[text.Length + 1];
        for (int i = 0; i < boundaries.Length; i++) boundaries[i] = i;
        return boundaries;
    }
    int[] prefix = new int[pattern.Length];
    for (int i = 1, q = 0; i < pattern.Length; i++)
    {
        while (q > 0 && pattern[i] != pattern[q]) q = prefix[q - 1];
        if (pattern[i] == pattern[q]) q++;
        prefix[i] = q;
    }
    var starts = new List<int>();
    for (int i = 0, q = 0; i < text.Length; i++)
    {
        while (q > 0 && text[i] != pattern[q]) q = prefix[q - 1];
        if (text[i] == pattern[q]) q++;
        if (q == pattern.Length)
        {
            starts.Add(i - pattern.Length + 1);
            q = prefix[q - 1];
        }
    }
    return starts.ToArray();
}
Console.WriteLine($"Overlaps: {string.Join(",", AllMatches("ABABAB", "ABAB"))}");
Console.WriteLine($"Repeated: {string.Join(",", AllMatches("AAAAA", "AAAA"))}");
Console.WriteLine($"Empty: {string.Join(",", AllMatches("AB", ""))}");


Overlaps: 0,2
Repeated: 0,1
Empty: 0,1,2


This focused version assumes non-null arguments and demonstrates the invariant, not comparison instrumentation. The full retained API below validates arguments and counts comparisons under its declared convention. Do not copy exact comparison counts from a differently instrumented loop.

Guided task: fill the q values for AAAA in AAAAA and circle the fallback immediately after each full match. Then process ABABAC while searching for ABABAB: follow the repeated border failures at C without changing the consumed text prefix. Finally compare every reported start with the direct matcher before discussing speed.

```{dropdown} Check the overlap and work arguments
The first four A characters reach q=4, report start zero, and fall back to three. The fifth A reaches four again and reports start one. For ABABAC against ABABAB, the final C fails against B at q=5, then B at q=3, then B at q=1, then A at zero. No full match is reported. Across the whole scan, q can increase only with consumed characters; fallback decreases q. Total decreases cannot exceed total increases, which bounds fallback work linearly as well as the forward scan.
```

For independent transfer, preserve the direct matcher's empty, overlapping and ordinal policies, reuse a prepared pattern across several texts, and include preparation plus reported-result storage in the workload account. The examples below retain the complete API, counted comparisons and larger boundary matrix.


**Common mistake.** Resetting `q` to zero after reporting a match. The occurrence of `ABAB` at start 2 in `ABABAB` is then missed, because the `AB` already matched at the end of the first occurrence is thrown away.


```{index} Reuse a Prefix Only When It Is Also the Needed Suffix
```

## Reuse a Prefix Only When It Is Also the Needed Suffix



KMP remembers how much of a pattern has already matched. After a mismatch, some of that matched text may also be a valid beginning of the next candidate match. The prefix table identifies such borders: proper prefixes that are also suffixes. Proper excludes the complete string itself, otherwise fallback would make no progress.

For pattern ABAB, the longest proper border of the whole pattern is AB, length two. For its prefixes A, AB, ABA and ABAB, the corresponding border lengths are 0, 0, 1 and 2. These are properties of the pattern; preparation can be reused when searching different texts under the same matching policy.

| Pattern prefix | Longest proper border | Length |
| --- | --- | --- |
| A | Empty | 0 |
| AB | Empty | 0 |
| ABA | A | 1 |
| ABAB | AB | 2 |

Suppose the matcher has recognized ABA of pattern ABAB and the next text code unit is C rather than the expected B. The matched prefix ABA has border A, of length one. It need not pretend that all recognized text is unrelated to the pattern. Falling back to a valid border keeps a suffix that already matches a prefix. If the new comparison also fails, repeat fallback through shorter borders. The text position can stay at the current unconsumed code unit while the matched-prefix length decreases.

For related explanation, see the further reading below.[^development-reading]


```{index} Separate Two Progress Arguments
```



```{index} Separate Two Progress Arguments
```

## Separate Two Progress Arguments



During scanning, either the text position advances or the matched-prefix length decreases through valid fallback links. A decrease cannot happen indefinitely without intervening increases. This accounting explains a linear number of scan comparisons under the algorithm's model; the proof is not simply that the outer loop mentions each text position once. The inner fallback work must be accounted for too.

Preparation costs work proportional to the pattern length. Scanning costs proportional to text length, with output storage proportional to the number of reported matches. When the same pattern is searched in many texts, an owned prepared table can avoid repeating preparation. If the table is exposed for arbitrary mutation, callers can break the matcher invariant; ownership/copying rules from the object chapters remain relevant.

After a full match, fallback must preserve the possibility of overlapping results. Resetting the matched length blindly to zero can lose occurrences such as ABA at starts zero and two in ABABA. Empty patterns need their explicit boundary contract before normal prefix-table indexing, because there is no pattern position zero to compare.

Guided exercise: build the prefix lengths for AAAA and explain why the final border has length three, not four. Trace the all-match scan on AAAAA. Compare the start positions with the direct matcher, then compare complete preparation-plus-scan work for one text and for several texts. Equivalent answers come before performance claims.


```{index} Proper Prefixes Survive a Mismatch
```

## Proper Prefixes Survive a Mismatch




```{index} Proper Prefixes Survive a Mismatch
```

prefix[i] is the longest proper prefix of pattern[0..i] that is also its suffix. Proper excludes the whole segment. For ababaca, the table is 0,0,1,2,3,0,1. If q units matched and the next comparison fails, only borders of the matched prefix can remain possible: set q=prefix[q−1] and try again.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var matcher=new PatternScan("ababaca");
Console.WriteLine("prefix="+string.Join(",",matcher.Prefix));
Console.WriteLine($"preparation comparisons={matcher.PreparationComparisons}");
var snapshot=matcher.Prefix;snapshot[2]=99;
Console.WriteLine("owned prefix="+string.Join(",",matcher.Prefix));
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


prefix=0,0,1,2,3,0,1
preparation comparisons=8
owned prefix=0,0,1,2,3,0,1


Prefix construction maintains a valid border length for the already processed prefix. On mismatch, following the previous border chain finds the next possible border rather than resetting blindly or skipping a valid candidate. Successful comparisons increase q by at most one; fallback decreases it. Across all positions the total fallback work is linear in m. The owned table is prepared once and can be reused across multiple texts.

**Common mistake.** Falling back to `prefix[q]` instead of `prefix[q - 1]`. The index is off by one, so the scan consults the border of a segment that includes the mismatched position and either skips a valid alignment or loops.


```{index} KMP Keeps the Text Moving Forward
```

## KMP Keeps the Text Moving Forward




```{index} KMP Keeps the Text Moving Forward
```

Before each comparison, q code units of the pattern equal the suffix of text already consumed. After a mismatch, fallback preserves that invariant without moving the text index backward. After a complete match, fall back to the pattern’s longest border so overlapping occurrences remain discoverable.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var matcher=new PatternScan("aaa");
foreach(string text in new[]{"aaaaa","baaab",""})
{
 var result=matcher.Search(text);
 Console.WriteLine($"text=[{text}], positions=[{string.Join(",",result.Positions)}], scan comparisons={result.Comparisons}");
}
Console.WriteLine("empty boundaries="+string.Join(",",new PatternScan("").Search("abc").Positions));
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


text=[aaaaa], positions=[0,1,2], scan comparisons=5
text=[baaab], positions=[1], scan comparisons=7
text=[], positions=[], scan comparisons=0
empty boundaries=0,1,2,3


Both preprocessing and scan use one counted equality test per candidate attempt, avoiding a hidden duplicate comparison in a while condition followed by an if. Preprocessing is O(m), each scan O(n), and the table O(m) space plus O(r) result storage. Total fallback decreases are bounded by total increases in matched length, which gives the linear bound despite the nested loop. Preparing for each request loses the benefit of repeated-pattern reuse.

**Common mistake.** Moving the text index back after a mismatch, as the direct scan does. That reintroduces the quadratic behavior KMP exists to avoid; the invariant guarantees that the consumed text never needs rereading.


```{index} Measure a Complete Workload and Define Text Units
```

## Measure a Complete Workload and Define Text Units




```{index} Measure a Complete Workload and Define Text Units
```

On repetitive text, direct scanning can repeatedly compare long matching prefixes. Include preparation in the KMP total, and reuse the same prepared pattern when comparing several texts. Operation counts are evidence, not timings.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

string text=new string('a',200)+"b",pattern=new string('a',20)+"b";
var direct=PatternScan.Direct(text,pattern);var matcher=new PatternScan(pattern);var kmp=matcher.Search(text);
if(!direct.Positions.SequenceEqual(kmp.Positions))throw new Exception("Matcher disagreement.");
Console.WriteLine($"positions={string.Join(",",kmp.Positions)}, direct={direct.Comparisons}, KMP prep={matcher.PreparationComparisons}, scan={kmp.Comparisons}, total={matcher.PreparationComparisons+kmp.Comparisons}");
string unicode="A😀B";
Console.WriteLine($"UTF-16 length={unicode.Length}, B index={new PatternScan("B").Search(unicode).Positions[0]}");
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


positions=180, direct=3801, KMP prep=39, scan=381, total=420
UTF-16 length=4, B index=3


UTF-16 offsets count code units, not user-perceived characters; a supplementary emoji takes two units. Exact ordinal matching performs no case folding, culture rules, or Unicode normalization. Canonically equivalent spellings can therefore differ. For ordinary application code, use .NET search APIs with an explicit comparison policy.[^2602_shortest_paths_strings-1] Do not claim this educational implementation outperforms the library without a controlled benchmark.

**Common mistake.** Reporting KMP as faster from the scan count alone. The prefix table costs comparisons too, and for a single short search the direct scan can be cheaper; the total column includes preparation for that reason.


```{index} Exercise
```

## Exercise




```{index} Exercise
```

Compare direct and KMP outputs for overlapping, empty, longer-than-text, and case-sensitive patterns. Check against ordinal library first-match results.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var cases=new[]{("aaaa","aa"),("abc",""),("ab","abcd"),("Audit","audit")};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

// Solution
var cases=new[]{("aaaa","aa"),("abc",""),("ab","abcd"),("Audit","audit")};
foreach(var pair in cases)
{
 var direct=PatternScan.Direct(pair.Item1,pair.Item2);var kmp=new PatternScan(pair.Item2).Search(pair.Item1);
 int first=kmp.Positions.Length==0?-1:kmp.Positions[0];
 if(!direct.Positions.SequenceEqual(kmp.Positions)||first!=pair.Item1.IndexOf(pair.Item2,StringComparison.Ordinal))throw new Exception("Ordinal mismatch.");
 Console.WriteLine($"[{pair.Item1}]/[{pair.Item2}]: positions=[{string.Join(",",kmp.Positions)}], first={first}");
}
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


[aaaa]/[aa]: positions=[0,1,2], first=0
[abc]/[]: positions=[0,1,2,3], first=0
[ab]/[abcd]: positions=[], first=-1
[Audit]/[audit]: positions=[], first=-1


The output lists `[0,1,2]` for `aa` in `aaaa`, the four boundaries `[0,1,2,3]` for the empty pattern, and no positions for a pattern longer than the text or one that differs only in case. Both matchers agree on every case and on the library's ordinal first match, which is the contract this reading set out to preserve.


```{rubric} Footnotes
```


[^2602_shortest_paths_strings-1]: Microsoft documents the comparison overloads and empty-pattern behavior of [String.IndexOf](https://learn.microsoft.com/en-us/dotnet/api/system.string.indexof) and recommends making comparison choices explicit in [string best practices](https://learn.microsoft.com/en-us/dotnet/standard/base-types/best-practices-strings).
[^2602_shortest_paths_strings-2]: KMP’s linear bound concerns code-unit comparisons and pattern preparation; returning every match also requires output storage.

[^development-reading]: Further reading: [Sedgewick and Wayne, Strings](https://algs4.cs.princeton.edu/50strings/). The worked business scenarios and guided reasoning here are original teaching examples.
